In [ ]:
import sympy as sp

A, B, PA, PB = sp.symbols("A B P_A P_B", real=True)
g, eps_d, kappa_b, n_b = sp.symbols(r"g \varepsilon_d \kappa_b n_b", real=True)

c = sp.sqrt(2) * g
A_0 = sp.sqrt(eps_d/g)
eta = 1+2*n_b

eq_a  = c * (PA * PB - A * B)
eq_pa = c * (B * PA - A * PB)
eq_b  = g * (A**2 + PA**2) / sp.sqrt(2) - eps_d / sp.sqrt(2) - kappa_b / 2 * B - kappa_b * (1 + 2*n_b) * PB
eq_pb = c * A * PA + kappa_b / 2 * PB

PB_sol = sp.solve(eq_pb, PB)[0]
B_sol  = sp.solve(eq_b.subs(PB, PB_sol), B)[0]
B_sol

In [ ]:
sp.expand(eq_a.subs(PB, PB_sol).subs(B, B_sol)*kappa_b/2/g**2)

In [ ]:
sp.expand(eq_pa.subs(PB, PB_sol).subs(B, B_sol)*kappa_b/2/g**2)

In [ ]:
Jac = sp.Matrix([[0, -c*A_0, 0, 0],[c*A_0, -kappa_b/2, 0, -kappa_b*(1+2*n_b)],[0, 0, 0, -c*A_0],[0, 0, c*A_0, kappa_b/2]])
for eig in Jac.eigenvects()[2:]:
    print(sp.simplify(eig[2][0][0]/eig[2][0][2]))

In [ ]:
eigenvec_unst = Jac.eigenvects()[2:]

eigenvec_unst[0][2][0][0]

In [ ]:
sp.simplify(eigenvec_unst[1][2][0][0])

In [ ]:
eigenvec_unst = Jac.eigenvects()[2:]

sp.simplify(eigenvec_unst[1][2][0][0].subs(sp.sqrt(eps_d/g), A_0)/eta)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

epsilon_p = 0.1
E_J = 37 * 2 * np.pi
phi_a = 0.11
phi_b = 0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 5 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2 *0
eps_d = 2 * alpha_sq * g
n_b = 2
eta = 1+2*n_b

A_all   = np.arange(-5.2, 5.2, 0.01)
PA_all  = np.arange(-5.2, 5.2, 0.01)
A, PA   = np.meshgrid(A_all, PA_all)

A_0   = np.sqrt(eps_d / g)
delta = kappa_a * kappa_b / 4 / g**2

H_vals = (-PA**2 * kappa_a / 2
          + ((A_0**2 - delta) * A * PA - A**3 * PA - A * PA**3 - 2 * (2*n_b+1) * A**2 * PA**2)
          * 2 * g**2 / kappa_b)

plt.figure()
plt.pcolormesh(A_all, PA_all, H_vals, cmap='RdBu_r')
plt.colorbar(label='Hamiltonian')
cs = plt.contour(A_all, PA_all, H_vals, levels=[0.0], colors='white', linewidths=2)
plt.scatter(np.sqrt(A_0**2 - delta), 0, color='black', s=10)
plt.xlabel('A')
plt.ylabel(r'$P_A$')
plt.tight_layout()
plt.show()

In [ ]:
paths = cs.get_paths()
contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)

A_contour  = contour_coords[:, 0]
PA_contour = contour_coords[:, 1]

tol=1e-6
A_0_p = np.sqrt(A_0**2-delta)
idx_keep = idx_keep = (A_contour > -tol) & (PA_contour > tol) & (PA_contour < A_0_p)

A_keep = A_contour[idx_keep]
PA_keep = PA_contour[idx_keep]


A_keep = np.insert(A_keep,0,A_0_p)
A_keep = np.append(A_keep,0)

PA_keep = np.insert(PA_keep,0,0)
PA_keep = np.append(PA_keep,0)

PB_keep = -2*np.sqrt(2)*g/kappa_b*A_keep*PA_keep
B_keep = np.sqrt(2)*g/kappa_b * (A_keep**2+4*(2*n_b+1)*A_keep*PA_keep+PA_keep**2-A_0_p**2) - kappa_a/2/np.sqrt(2)/g

B_th = np.linspace(0,2)
P_B_th = -B_th/eta
P_A_th = np.linspace(0,4)
A_th = -eta*P_A_th+np.sqrt((eta**2-1)*P_A_th**2+A_0**2)

fig, axes = plt.subplots(1,2)
ax_1, ax_2 = axes

ax_1.scatter(A_keep,PA_keep, c=range(len(A_keep)))
ax_1.plot(A_th, P_A_th, "r--")
ax_1.set_xlabel(r'$A$')
ax_1.set_ylabel(r'$P_A$')

sc = ax_2.scatter(B_keep, PB_keep, c=range(len(B_keep)))
ax_2.plot(B_th, P_B_th, "r--")
ax_2.set_xlabel(r'$B$')
ax_2.set_ylabel(r'$P_B$')

plt.colorbar(sc, ax=ax_2)
plt.tight_layout()
plt.show()

In [ ]:
dA = np.diff(A_keep)
dPA = np.diff(PA_keep)
dB = np.diff(B_keep)
dPB = np.diff(PB_keep)

action = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:]+dB*PB_keep[1:]-dPB*B_keep[1:])
action

In [ ]:
A_0**2*np.log(eta+np.sqrt(eta**2-1))/np.sqrt(eta**2-1)

In [ ]:
n_b_ratio_lim = 5
#kappa_a_list = np.logspace(np.log10(kappa_a_ratio_lim)-1e-1,-4,20)[:]*kappa_a
n_b_list = np.linspace(0.1,5,100)
action_list=[]
action_A_only_list=[]
action_th_list=[]
plot_path=False
for n_b_i in n_b_list:  
  eta_i = 1+2*n_b_i

  H_vals = (-PA**2 * kappa_a / 2
    + ((A_0**2 - delta) * A * PA - A**3 * PA - A * PA**3 - 2 * (2*n_b_i+1) * A**2 * PA**2)
    * 2 * g**2 / kappa_b)

  fig_dummy, ax_dummy = plt.subplots()
  cs = ax_dummy.contour(A_all, PA_all, H_vals, levels=[0.0])
  plt.close(fig_dummy)
  paths = cs.get_paths()
  contour_coords = np.concatenate([p.vertices for p in paths], axis=0)  # shape (N, 2)

  A_contour  = contour_coords[:, 0]
  PA_contour = contour_coords[:, 1]
  tol=1e-6
  A_0_p = np.sqrt(A_0**2-delta)
  idx_keep = idx_keep = (A_contour > -tol) & (PA_contour > tol) & (PA_contour < A_0_p)

  A_keep = A_contour[idx_keep]
  PA_keep = PA_contour[idx_keep]


  A_keep = np.insert(A_keep,0,A_0_p)
  A_keep = np.append(A_keep,0)

  PA_keep = np.insert(PA_keep,0,0)
  PA_keep = np.append(PA_keep,0)

  PB_keep = -2*np.sqrt(2)*g/kappa_b*A_keep*PA_keep
  B_keep = np.sqrt(2)*g/kappa_b * (A_keep**2+4*(2*n_b_i+1)*A_keep*PA_keep+PA_keep**2-A_0_p**2) - kappa_a/2/np.sqrt(2)/g
  dA = np.diff(A_keep)
  dPA = np.diff(PA_keep)
  dB = np.diff(B_keep)
  dPB = np.diff(PB_keep)

  action = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:]+dB*PB_keep[1:]-dPB*B_keep[1:])
  action_list.append(action)
  action_A_only = -np.sum(dA*PA_keep[1:]-dPA*A_keep[1:])
  action_A_only_list.append(action_A_only)

  action_th = A_0**2*np.log(eta_i+np.sqrt(eta_i**2-1))/np.sqrt(eta_i**2-1)
  action_th_list.append(action_th)

  if plot_path:
    B_th = np.linspace(0,2)
    P_B_th = -B_th/eta_i
    P_A_th = np.linspace(0,4)
    A_th = -eta_i*P_A_th+np.sqrt((eta_i**2-1)*P_A_th**2+A_0**2)    
    fig, axes = plt.subplots(1,2)
    ax_1, ax_2 = axes

    ax_1.scatter(A_keep,PA_keep, c=range(len(A_keep)))
    ax_1.plot(A_th, P_A_th, "r--")
    ax_1.set_xlabel(r'$A$')
    ax_1.set_ylabel(r'$P_A$')

    sc = ax_2.scatter(B_keep, PB_keep, c=range(len(B_keep)))
    ax_2.plot(B_th, P_B_th, "r--")
    ax_2.set_xlabel(r'$B$')
    ax_2.set_ylabel(r'$P_B$')

    plt.colorbar(sc, ax=ax_2)
    plt.tight_layout()
    plt.show()

action_list = np.array(action_list)
action_A_only_list = np.array(action_A_only_list)
action_th_list = np.array(action_th_list)

In [ ]:
plt.plot(n_b_list, -action_list, label="Adiabatic limit", lw=4)
plt.plot(n_b_list, -action_A_only_list, label="Mode a only")
plt.plot(n_b_list, -action_th_list, "r--", label="Theory")

plt.xlabel(r"$n_b$"+" (MHz)")
plt.ylabel(r"$S$")
plt.legend()
plt.show()